<a href="https://colab.research.google.com/github/SAILResearch/AI_Teammates_in_SE3/blob/main/analysis/load_AIDev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### This can be used in the motivation: generating code that goes against coding practices or project guidelines:
https://github.com/evstack/ev-node/pull/2388

## AIDev dataset

In [1]:
import pandas as pd

from __future__ import annotations

!pip install cliffs-delta

from cliffs_delta import cliffs_delta
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Dict, Any, Tuple

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

from datetime import datetime

FONT_SCALE = 1.10  # ← master knob
_BASE_FONT = 10

SECONDS_TO_HOUR = 3600
SECONDS_TO_DAY = 86_400

sns.set_style("whitegrid")
sns.set_context("talk", font_scale=FONT_SCALE)

plt.rcParams.update({
      # 'font.family': 'serif',
      'font.serif': ['Times New Roman', 'Times'],
      'mathtext.fontset': 'stix',
      'axes.linewidth': 1.0,
      'axes.edgecolor': '#000000',    
      'axes.labelsize': _BASE_FONT,
      'axes.titlesize': _BASE_FONT,
      'xtick.major.size': 2,
      'ytick.major.size': 2,
      'xtick.labelsize': _BASE_FONT,
      'ytick.labelsize': _BASE_FONT,
      'legend.fontsize': _BASE_FONT,
      'legend.title_fontsize': _BASE_FONT,
      'axes.edgecolor': 'grey',
      'figure.dpi': 300,
})

try:
  from helper import (  # type: ignore
      FIG_DIR,
      CORR_DIR,
      COLOR_MAP,
      NAME_MAPPING,
      FLOW_ORDER,
      mannUandCliffdelta,  # statistical tests
  )
except Exception:
  FIG_DIR = Path("figs")
  FIG_DIR.mkdir(exist_ok=True)
  CORR_DIR = Path("corr")
  CORR_DIR.mkdir(exist_ok=True)
  COLOR_MAP = {
    "Human": "#56B4E9",
    "OpenAI_Codex": "#D55E00",
    "OpenAI Codex": "#D55E00",
    "Codex": "#F0E442",
    "Devin": "#009E73",
    "Copilot": "#0072B2",
    "GitHub Copilot": "#0072B2",
    "Cursor": "#785EF0",
    "Claude_Code": "#DC267F",
    "Claude Code": "#DC267F",
  }
  NAME_MAPPING = {
    "OpenAI_Codex": "OpenAI Codex",
    "Codex": "OpenAI Codex",
    "Devin": "Devin",
    "Copilot": "GitHub Copilot",
    "Cursor": "Cursor",
    "Claude_Code": "Claude Code",
    "Claude": "Claude Code",
    "Human": "Human",
    "OpenAI Codex": "OpenAI Codex",
    "GitHub Copilot": "GitHub Copilot",
    "Claude Code": "Claude Code",
  }
  for key, value in NAME_MAPPING.items():
      if key not in COLOR_MAP:
          COLOR_MAP[key] = COLOR_MAP[value]
  FLOW_ORDER = [
    "feat",  # new functionality
    "fix",  # bug fixes
    "perf",  # performance work
    "refactor",  # code reshaping
    "style",  # lint / formatting
    "docs",  # documentation
    "test",  # testing
    "chore",  # misc maintenance
    "build",  # packaging / build sys
    "ci",  # continuous-integration
    "other",  # continuous-integration
  ]

  def mannUandCliffdelta(dist1, dist2):
    d, size = cliffs_delta(dist1, dist2)
    print(f"Cliff's delta: {size}, d={d}")
    u, p = stats.mannwhitneyu(dist1, dist2, alternative="two-sided")
    print(f"Mann-Whitney-U-test: u={u} p={p}")
    return u, p, d, size


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


#### 3. Read PRs and Repos

In [11]:
def generate_repo_stats_table(df: pd.DataFrame):
    """
    Generate a LaTeX table with repository statistics from GitHub PR data.
'', ''
    Args:
        df (pd.DataFrame): DataFrame containing GitHub PR data with repository metadata
        output_file (str): Path to output LaTeX file
    """
    # Step 1: Remove duplicate repositories (keep first occurrence)
    df_unique = df.sort_values("stargazerCount", ascending=0).drop_duplicates(
        subset=['repo'], keep='first').copy()

    # Count security PRs per repository
    agentic_prs = df.groupby('repo').size()
    df_unique['agentic_pr_count'] = df_unique['repo'].map(agentic_prs).fillna(0)

    # Step 3: Prepare the statistics table
    metrics = {
        'Commits': 'commitCount',
        'Agentic PRs': 'agentic_pr_count',
        'Stars': 'stargazerCount',
        # 'Issues': 'issue_count',
        # 'Users': 'mentionableUsersCount'
    }

    # Calculate statistics for each metric
    stats_data = []
    for metric_name, col_name in metrics.items():
        values = df_unique[col_name]
        stats_data.append({
            'Metric': metric_name,
            'Min.': np.min(values),
            'Median': np.median(values),
            'Mean': np.mean(values),
            'Max.': np.max(values)
        })

    # Create stats DataFrame
    stats_df = pd.DataFrame(stats_data)
    stats_df['Max.'] = stats_df['Max.'].apply(lambda x: f"{x:,}")
    stats_df['Median'] = stats_df['Median'].apply(lambda x: f"{x:,.2f}")
    stats_df['Mean'] = stats_df['Mean'].apply(lambda x: f"{x:,.2f}")

    # Step 4: Generate LaTeX table
    latex_table = stats_df.to_latex(
        index=False,
        column_format='lrrrr',
        float_format="{:0.2f}".format,
        caption=f'Statistics of the {df['repo'].nunique():,} studied JavaScript projects.',
        label='tab:repo-stats',
        position='t!'
    )
    latex_table = latex_table.replace(".00", "")
    # Save to file
    # with open(output_file, 'w') as f:
    #     f.write(latex_table)

    return latex_table, df_unique

In [16]:
pr_df = pd.read_csv("../data/prs_features.csv")
# pr_df = pr_df[pr_df['agent_configured'] == True]

In [17]:
pr_df['repo'].nunique(), len(pr_df)

(2211, 58127)

In [20]:
# Select project with at least 5 projects before the first intruction files and 5 PRs after
agent_compar_df1 = pd.read_csv("../data/repo_agent_metrics_comparison_mode_1.csv")
projects = agent_compar_df1.loc[
    (agent_compar_df1['pr_count_after']>=5) &
    (agent_compar_df1['pr_count_before']>=5),
    'repo'
].unique()

# latex_table, df_unique = generate_repo_stats_table(
#     pr_df[
#         (pr_df['agent_configured'] == True) &
#         (pr_df['repo'].isin(projects))
#     ]
# )

### RQ1

In [21]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_merge_rate_barplot(df, filename=None):
    # --- FIX: Create an explicit copy to avoid SettingWithCopyWarning ---
    df = df.copy()

    # --- 1. Define Categories Logic ---
    df['category'] = None

    # A. Handle Exact Zeros
    df.loc[df['merge_impro_pct'] == 0, 'category'] = '0%'

    # B. Handle Positives (> 0)
    pos_mask = df['merge_impro_pct'] > 0
    pos_bins = [0, 20, 40, 60, 80, 100, np.inf]
    pos_labels = ["0–20%", "20–40%", "40–60%", "60–80%", "80–100%", "≥100%"]
    
    df.loc[pos_mask, 'category'] = pd.cut(
        df.loc[pos_mask, 'merge_impro_pct'], 
        bins=pos_bins, labels=pos_labels, right=False
    )
    
    df.loc[np.isinf(df['merge_impro_pct']) & (df['merge_impro_pct'] > 0), 'category'] = "≥100%"

    # C. Handle Negatives (< 0)
    neg_mask = df['merge_impro_pct'] < 0
    neg_bins = [-np.inf, -100, -80, -60, -40, -20, 0]
    neg_labels = ["≤-100%", "-100–-80%", "-80–-60%", "-60–-40%", "-40–-20%", "-20–0%"]
    
    df.loc[neg_mask, 'category'] = pd.cut(
        df.loc[neg_mask, 'merge_impro_pct'], 
        bins=neg_bins, labels=neg_labels, right=True
    )

    # --- 2. Aggregate and Sort ---
    ordered_categories = neg_labels + ['0%'] + pos_labels
    cat_counts = df['category'].value_counts().reindex(ordered_categories, fill_value=0)

    plot_df = pd.DataFrame({
        "range_label": cat_counts.index,
        "count": cat_counts.values
    })

    # --- 3. Plot ---
    plt.figure(figsize=(4, 2.65))

    ax = sns.barplot(
        data=plot_df, 
        x="range_label", 
        y="count", 
        edgecolor="none", 
        color='grey'
    )

    # --- Add labels above bars ---
    ax.bar_label(ax.containers[0], padding=2.75, fmt='%d', fontsize=8)

    # Increase y-limit to ensure labels fit
    plt.ylim(0, plot_df['count'].max() * 1.12)

    plt.xlabel("Merge rate improvement", labelpad=0) # Pulls the axis title closer
    plt.ylabel("# of projects")

    # --- FIX: Rotate 45 degrees and pull close to ticks ---
    plt.xticks(
        rotation=45, 
        ha='right',            # Align the end of the text to the tick
        rotation_mode='anchor', # Rotates the text around the alignment point
        fontsize=9
    )

    # Use a small pad_inches for the savefig to keep it tight for the paper
    plt.tight_layout()
    if filename:
        plt.savefig(f'../figs/{filename}', bbox_inches='tight', pad_inches=0.01)
        plt.close()
    else:
        plt.show()

##### 1. PRs closed before versus after the introduction of the first custom instruction file

In [24]:
agent_compar_df1 = pd.read_csv("../data/repo_agent_metrics_comparison_mode_1.csv")
agent_compar_df1 = agent_compar_df1[
    (agent_compar_df1['pr_count_after']>=5) &
    (agent_compar_df1['pr_count_before']>=5)
]

df = plot_merge_rate_barplot(
    agent_compar_df1,
    filename='barplot_1.pdf'
)

##### Percent of projects that increased their merge rate by at least 20%

In [25]:
len(agent_compar_df1[agent_compar_df1['merge_impro_pct']>=20])/len(agent_compar_df1), len(agent_compar_df1[agent_compar_df1['merge_impro_pct']<=-20])/len(agent_compar_df1)

(0.3191489361702128, 0.1595744680851064)

In [6]:
agent_compar_df1.loc[agent_compar_df1['merge_impro_pct']>=20, ['repo', 'merge_impro_pct', 'merge_rate_before', 'merge_rate_after']]

,repo,merge_impro_pct,merge_rate_before,merge_rate_after
5,dotnet/runtime,106.62,0.333333,0.688720
6,dotnet/roslyn,70.76,0.444444,0.758929
44,ant-design/ant-design,162.30,0.200000,0.524590
46,openshift/console,29.25,0.700000,0.904762
62,envoyproxy/envoy,100.00,0.125000,0.250000
67,ethereum-optimism/optimism,52.54,0.500000,0.762712
93,continuedev/continue,50.00,0.400000,0.600000
102,steedos/steedos-platform,30.91,0.555556,0.727273
109,cgeo/cgeo,169.23,0.142857,0.384615
118,dotnet/maui,inf,0.000000,0.417582


In [7]:
agent_compar_df1.loc[agent_compar_df1['merge_impro_pct']<=-20, ['repo', 'merge_impro_pct', 'merge_rate_before', 'merge_rate_after']]

,repo,merge_impro_pct,merge_rate_before,merge_rate_after
16,grafana/grafana,-22.09,1.000000,0.779141
104,wasmerio/wasmer,-20.00,0.250000,0.200000
131,langflow-ai/langflow,-36.65,0.676471,0.428571
161,risingwavelabs/risingwave,-72.73,1.000000,0.272727
170,datahub-project/datahub,-38.59,0.875000,0.537367
190,apache/datafusion,-51.32,0.725000,0.352941
197,coder/coder,-35.12,0.410256,0.266187
199,chainapsis/keplr-wallet,-21.74,0.958333,0.750000
274,thirdweb-dev/js,-25.42,1.000000,0.745763
317,kubernetes-sigs/headlamp,-100.00,0.166667,0.000000


##### Percent of projects that reduce their time to merge

In [7]:
len(agent_compar_df1[agent_compar_df1['merge_time_delta']>0])/len(agent_compar_df1)

0.40119760479041916

### RQ2

##### Load the dataset

In [26]:
agent_compar_merged_df1 = pd.read_csv("../data/repo_agent_metrics_comparison_mode_3.csv")
agent_compar_merged_df1 = agent_compar_merged_df1[
    (agent_compar_merged_df1['pr_count_after']>=5) &
    (agent_compar_merged_df1['pr_count_before']>=5)
]

In [27]:
len(agent_compar_merged_df1)

126

##### Percent of projects (at least 5 merged PRs before the first inst file and 5 merged PRs after the same file) that managed to significantly increase all agentic PR complexity metrics

In [29]:
num_projects = len(agent_compar_merged_df1)

increased_code_churn = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['code_churn_delta']<0)&
    # (agent_compar_merged_df1['code_churn_eff_size']!="negligible")&
    (agent_compar_merged_df1['code_churn_p_value']<0.05)
])/num_projects
increased_description_length = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['description_length_delta']<0)&
    # (agent_compar_merged_df1['description_length_eff_size']!="negligible")&
    (agent_compar_merged_df1['description_length_p_value']<0.05)
])/num_projects
increased_num_commits = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['num_commits_delta']<0)&
    # (agent_compar_merged_df1['num_commits_eff_size']!="negligible")&
    (agent_compar_merged_df1['num_commits_p_value']<0.05)
])/num_projects
increased_num_files = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['num_files_delta']<0)&
    # (agent_compar_merged_df1['num_files_eff_size']!="negligible")&
    (agent_compar_merged_df1['num_files_p_value']<0.05)
])/num_projects

increased_description_length, increased_code_churn, increased_num_commits, increased_num_files

(0.20634920634920634,
 0.1349206349206349,
 0.0873015873015873,
 0.10317460317460317)

In [12]:
agent_compar_merged_df1.loc[
    (agent_compar_merged_df1['code_churn_delta']<0)&
    (agent_compar_merged_df1['code_churn_p_value']<0.05),
    ['repo', 'code_churn_before', 'code_churn_after', 'code_churn_eff_size', 'description_length_before', 'description_length_after', 'description_length_eff_size', 'num_commits_before', 'num_commits_after', 'num_commits_eff_size', 'num_files_before', 'num_files_after', 'num_files_eff_size']
]

,repo,code_churn_before,code_churn_after,code_churn_eff_size,description_length_before,description_length_after,description_length_eff_size,num_commits_before,num_commits_after,num_commits_eff_size,num_files_before,num_files_after,num_files_eff_size
130,qoretechnologies/qore,1970.5,5546.0,medium,1220.0,1498.0,small,3.0,6.5,small,16.0,42.0,medium
146,vllm-project/vllm,61.0,1050.0,medium,1070.0,1829.0,small,2.0,3.0,medium,3.5,37.5,small
175,chainapsis/keplr-wallet,66.0,338.0,medium,508.0,707.0,small,2.0,3.5,medium,2.0,6.5,small
223,sgl-project/sglang,6.5,101.0,medium,1333.5,965.0,negligible,1.0,2.0,small,1.0,3.0,small
325,dartsim/dart,5.0,154.5,large,180.0,306.0,large,1.0,3.0,large,1.0,3.5,medium
360,epam/ketcher,36.5,148.5,medium,3771.5,3619.5,negligible,3.0,4.0,small,3.0,4.0,negligible
376,microsoft/testfx,113.0,391.5,medium,2311.0,2592.0,negligible,4.0,7.0,small,3.0,13.0,medium
408,tambo-ai/tambo,104.0,605.0,medium,915.0,1093.0,medium,2.0,4.0,negligible,3.0,8.0,medium
467,bigcapitalhq/bigcapital,30.0,699.0,medium,1320.0,1230.5,negligible,1.0,1.0,negligible,8.0,9.0,negligible
512,Autonomy-Logic/openplc-editor,23.0,319.0,large,1601.0,1490.0,negligible,1.0,1.0,small,3.0,4.0,medium


##### Percent of projects that managed to significantly decrease all agentic PR complexity metrics

In [30]:
num_projects = len(agent_compar_merged_df1)

increased_code_churn = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['code_churn_delta']>0)&
    # (agent_compar_merged_df1['code_churn_eff_size']!="negligible")&
    (agent_compar_merged_df1['code_churn_p_value']<0.05)
])/num_projects
increased_description_length = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['description_length_delta']>0)&
    # (agent_compar_merged_df1['description_length_eff_size']!="negligible")&
    (agent_compar_merged_df1['description_length_p_value']<0.05)
])/num_projects
increased_num_commits = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['num_commits_delta']>0)&
    # (agent_compar_merged_df1['num_commits_eff_size']!="negligible")&
    (agent_compar_merged_df1['num_commits_p_value']<0.05)
])/num_projects
increased_num_files = len(agent_compar_merged_df1[
    (agent_compar_merged_df1['num_files_delta']>0)&
    # (agent_compar_merged_df1['num_files_eff_size']!="negligible")&
    (agent_compar_merged_df1['num_files_p_value']<0.05)
])/num_projects

increased_description_length, increased_code_churn, increased_num_commits, increased_num_files

(0.10317460317460317,
 0.07142857142857142,
 0.0873015873015873,
 0.07936507936507936)

In [10]:
agent_compar_merged_df1.loc[
    (agent_compar_merged_df1['description_length_delta']>0)&
    (agent_compar_merged_df1['description_length_p_value']<0.05),
    ['repo', 'description_length_before', 'description_length_after', 'description_length_eff_size', 'code_churn_before', 'code_churn_after', 'code_churn_eff_size', 'num_commits_before', 'num_commits_after', 'num_commits_eff_size', 'num_files_before', 'num_files_after', 'num_files_eff_size']
]

,repo,description_length_before,description_length_after,description_length_eff_size,code_churn_before,code_churn_after,code_churn_eff_size,num_commits_before,num_commits_after,num_commits_eff_size,num_files_before,num_files_after,num_files_eff_size
43,apache/doris,1551.0,1101.0,large,580.0,487.5,negligible,2.0,2.0,negligible,6.0,5.0,small
194,OpenAgentsInc/openagents,2083.0,980.5,large,1560.0,1086.0,small,4.0,2.0,small,11.0,8.0,small
320,elsa-workflows/elsa-core,2981.5,1680.0,medium,87.0,64.0,negligible,2.0,2.0,small,2.0,2.0,negligible
321,OpenCCU/OpenCCU,3227.0,1309.0,large,136.0,24.5,small,3.0,2.0,small,4.0,1.0,medium
391,CherryHQ/cherry-studio,2665.0,2152.0,medium,131.0,116.0,negligible,2.0,2.0,small,6.0,4.0,negligible
440,unpoller/unpoller,2349.0,1260.5,large,24.0,103.0,large,1.0,1.0,small,3.0,6.0,large
556,lance-format/lance,1029.0,570.0,medium,433.0,449.0,negligible,4.0,3.0,negligible,4.0,5.5,negligible
633,workleap/wl-hopper,1692.5,824.0,large,653.0,268.0,large,3.0,2.0,negligible,7.5,7.0,negligible
882,openvm-org/openvm,1430.0,630.0,large,247.5,348.0,small,4.0,2.0,small,5.5,9.0,small
901,miurla/morphic,1488.0,740.0,large,101.0,40.0,small,2.0,2.0,negligible,3.0,2.0,small


##### Percent of projects that decreased their merge rate by at least 20%

In [31]:
len(agent_compar_df1[agent_compar_df1['merge_impro_pct']<=-20])#/len(agent_compar_df1)

30

##### Anayzing other metrics

In [32]:
metric = 'description_length'
item_negative_df = agent_compar_df1.loc[
    (agent_compar_df1[f'{metric}_delta']<0) &
    (agent_compar_df1[f'{metric}_eff_size'].isin(['small', 'medium', 'large'])),
    [f'{metric}_before', f'{metric}_after', f'{metric}_delta', f'{metric}_p_value', f'{metric}_eff_size']
]

In [33]:
item_negative_df[(item_negative_df[f'{metric}_p_value']<0.05)&(item_negative_df[f'{metric}_eff_size']=='large')]

,description_length_before,description_length_after,description_length_delta,description_length_p_value,description_length_eff_size
46,838.5,2160.0,-0.558730,4.867578e-03,large
78,2619.0,5172.0,-0.590916,1.213873e-07,large
102,1899.0,3224.0,-0.737374,6.237007e-03,large
118,1225.0,2629.0,-0.638540,3.818882e-03,large
124,1119.0,1498.0,-0.667652,4.156986e-09,large
187,492.0,3980.0,-0.988511,9.889979e-41,large
197,553.0,1280.0,-0.717949,7.829327e-12,large
246,1558.0,4169.0,-0.723077,6.462199e-03,large
274,1339.0,2542.0,-0.850847,1.765565e-03,large
296,3873.0,6698.0,-0.714286,4.797980e-02,large


##### 2. PRs closed before the first versus after the last instruction file

In [34]:
agent_compar_df2 = pd.read_csv("../data/repo_agent_metrics_comparison_mode_2.csv")
agent_compar_df2 = agent_compar_df2[
    (agent_compar_df2['pr_count_before']>=5) &
    (agent_compar_df2['pr_count_after_last']>=5)
]

plot_merge_rate_barplot(
    agent_compar_df2,
    filename='barplot_2.pdf'
)

In [35]:
agent_compar_df2[['repo', 'merge_time_before', 'merge_time_after_last']]

,repo,merge_time_before,merge_time_after_last
5,dotnet/runtime,9.595833,64.665833
6,dotnet/roslyn,138.120972,44.161389
16,grafana/grafana,3.005000,32.533056
30,unoplatform/uno,2.857500,24.225556
41,vercel/next.js,11.865000,25.152500
...,...,...,...
1585,github-copilot-resources/copilot-metrics-viewer,15.086806,2.292500
1615,matin/garth,0.225556,0.182778
1636,openai/openai-dotnet,39.079444,17.521528
1643,enkan/enkan,0.288611,0.296944


In [36]:
len(agent_compar_df2[agent_compar_df2['merge_impro_pct']>=20])/len(agent_compar_df2), len(agent_compar_df2[agent_compar_df2['merge_impro_pct']<=-20])/len(agent_compar_df2)

(0.3236994219653179, 0.1676300578034682)

In [34]:
agent_compar_df2.loc[agent_compar_df2['merge_impro_pct']>=20, ['repo', 'merge_impro_pct', 'merge_rate_before', 'merge_rate_after_last']]

,repo,merge_impro_pct,merge_rate_before,merge_rate_after_last
5,dotnet/runtime,106.62,0.333333,0.688720
6,dotnet/roslyn,67.69,0.444444,0.745283
59,envoyproxy/envoy,100.00,0.125000,0.250000
89,continuedev/continue,50.00,0.400000,0.600000
96,sanity-io/sanity,39.68,0.437500,0.611111
98,steedos/steedos-platform,30.91,0.555556,0.727273
105,cgeo/cgeo,169.23,0.142857,0.384615
114,dotnet/maui,inf,0.000000,0.160000
123,azerothcore/azerothcore-wotlk,inf,0.000000,0.937500
126,kovidgoyal/kitty,inf,0.000000,0.212766


In [37]:
agent_compar_df2.loc[agent_compar_df2['merge_impro_pct']<=-20, ['repo', 'merge_impro_pct', 'merge_rate_before', 'merge_rate_after_last']]

,repo,merge_impro_pct,merge_rate_before,merge_rate_after_last
16,grafana/grafana,-27.27,1.000000,0.727273
104,wasmerio/wasmer,-20.00,0.250000,0.200000
131,langflow-ai/langflow,-36.65,0.676471,0.428571
161,risingwavelabs/risingwave,-72.73,1.000000,0.272727
190,apache/datafusion,-51.32,0.725000,0.352941
197,coder/coder,-73.25,0.410256,0.109756
199,chainapsis/keplr-wallet,-21.74,0.958333,0.750000
222,vercel/vercel,-20.59,0.666667,0.529412
274,thirdweb-dev/js,-26.79,1.000000,0.732143
486,cline/cline,-39.09,0.328358,0.200000


In [4]:
negative_merge_rate_df = agent_compar_df2.loc[agent_compar_df2['merge_impro_pct']<=-10, ['repo', 'merge_impro_pct', 'merge_rate_before', 'merge_rate_after_last', 'num_agent_files']]
negative_merge_rate_df['merge_type'] = 'Negative'
positive_merge_rate_df = agent_compar_df2.loc[agent_compar_df2['merge_impro_pct']>=10, ['repo', 'merge_impro_pct', 'merge_rate_before', 'merge_rate_after_last', 'num_agent_files']]
positive_merge_rate_df['merge_type'] = 'Positive'

combined_df = pd.concat((negative_merge_rate_df, positive_merge_rate_df))
combined_df.to_excel("../data/pos_neg_merge_rate_projects.xlsx", index=None)

#### RQ3

In [38]:
agent_compar_df3 = pd.read_csv("../data/repo_agent_metrics_comparison_mode_3.csv")
agent_compar_df3 = agent_compar_df3[
    (agent_compar_df3['pr_count_after']>=5) &
    (agent_compar_df3['pr_count_before']>=5)
]

In [40]:
num_projects = len(agent_compar_df3)

decreased_merge_time = len(agent_compar_df3[
    (agent_compar_df3['merge_time_delta']>0)&
    # (agent_compar_df3['merge_time_eff_size']!='negligible')&
    (agent_compar_df3['merge_time_p_value']<0.05)
])/num_projects

decreased_comments = len(agent_compar_df3[
    (agent_compar_df3['comments_delta']>0)&
    # (agent_compar_df3['comments_eff_size']!='negligible')&
    (agent_compar_df3['comments_p_value']<0.05)
])/num_projects

increased_merge_time = len(agent_compar_df3[
    (agent_compar_df3['merge_time_delta']<0)&
    # (agent_compar_df3['merge_time_eff_size']!='negligible')&
    (agent_compar_df3['merge_time_p_value']<0.05)
])/num_projects

increased_comments = len(agent_compar_df3[
    (agent_compar_df3['comments_delta']<0)&
    # (agent_compar_df3['comments_eff_size']!='negligible')&
    (agent_compar_df3['comments_p_value']<0.05)
])/num_projects


increased_merge_time, increased_comments, decreased_merge_time, decreased_comments

(0.1349206349206349,
 0.15079365079365079,
 0.07142857142857142,
 0.1111111111111111)

##### Percent of projects that managed to significantly increase the review efforts in terms of merge time and discussion size

In [42]:
num_projects = len(agent_compar_df3)

increased_merge_time = len(agent_compar_df3[
    (agent_compar_df3['merge_time_delta']<0)&
    # (agent_compar_df3['merge_time_eff_size']!='negligible')&
    (agent_compar_df3['merge_time_p_value']<0.05)
])/num_projects

increased_comments = len(agent_compar_df3[
    (agent_compar_df3['comments_delta']<0)&
    # (agent_compar_df3['comments_eff_size']!='negligible')&
    (agent_compar_df3['comments_p_value']<0.05)
])/num_projects

increased_merge_time, increased_comments

(0.1349206349206349, 0.15079365079365079)

In [39]:
metric = 'comments'
item_df = agent_compar_df3.loc[
    (agent_compar_df3[f'{metric}_delta']<0) &
    (agent_compar_df3[f'{metric}_eff_size'].isin(['small', 'medium', 'large'])),
    ['repo', f'{metric}_before', f'{metric}_after', f'{metric}_delta', f'{metric}_p_value', f'{metric}_eff_size']
]

In [40]:
pr_df.loc[
    (pr_df['repo'].str.endswith('boxwise/boxtribute')) &
    (pr_df['state']=='merged') &
    (pr_df['closed_at']>'2025-09-02T10:11:43'),
    'comments'
].describe()

count    35.000000
mean      2.657143
std       2.496384
min       1.000000
25%       1.000000
50%       2.000000
75%       3.000000
max      13.000000
Name: comments, dtype: float64

#### 4. Compare project word count between project with merge rate less than -10% and project with more than 10%

In [43]:
columns = ['repo', 'word_count', 'content_length', 'merge_impro_pct', 'H1', 'H2', 'H3', 'H4', 'H5', 'H6']

In [44]:
df1 = agent_compar_df1[columns].copy()
df2 = agent_compar_df2[columns].copy()
# df2 = agent_compar_df2[columns]

# df = pd.concat((df1, df2))
# df = df.drop_duplicates(subset=['repo'], keep='last')

def label_merge_rate_impro(x, threshold=20):
    if x <= -threshold:
        return 'Decrease'
    if x >= threshold:
        return 'Increase'
    return 'normal'

df1['label'] = df1['merge_impro_pct'].map(label_merge_rate_impro)
df2['label'] = df2['merge_impro_pct'].map(label_merge_rate_impro)

df1 = df1[df1['label']!='normal']
df2 = df2[df2['label']!='normal']

df1.reset_index(drop=True, inplace=True)
df2.reset_index(drop=True, inplace=True)

In [45]:
agent_compar_df4 = pd.read_csv("../data/repo_agent_metrics_comparison_mode_4.csv")
agent_compar_df4 = agent_compar_df4[
    (agent_compar_df4['pr_count_after_last']>=5) &
    (agent_compar_df4['pr_count_before']>=5)
]

In [46]:
len(agent_compar_df4)

118

##### Plot boxplot for the the word count and headings

In [47]:
def plot_boxplot(
    data,
    x,
    y,
    xlab,
    ylab,
    hue,
    median_order,
    median_values,
    file=None,
    log_scale=True,
    y_ticks=None,
    ylim=None,
    median_size=7,
    median_offset=3  # <-- New parameter: defaults to 3 points
):
    # Create figure
    fig, ax = plt.subplots(figsize=(3.5, 2.3))

    sns.boxplot(
        data=data,
        x=x,
        y=y,
        order=median_order,
        showfliers=False,
        showcaps=False,
        patch_artist=True,
        palette='Set2',
        hue=hue,
        legend=False,
        medianprops={"linewidth": 1, "color": "grey"},
        width=0.6,
        ax=ax
    )

    plt.xlabel(xlab)
    plt.ylabel(ylab)

    if ylim:
      ax.set_ylim(ylim)

    if log_scale:
        ax.set_yscale("symlog")
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_ticks)
    elif y_ticks:
        ax.set_yticks(y_ticks)

    # --- ADJUSTMENT: Use the median_offset parameter ---
    for i, category in enumerate(median_order):
        if category in median_values.index:
            median_val = median_values.loc[category]

            text_an = f"{median_val:.2f}".replace(".00", "")
            ax.annotate(
                text=text_an,
                xy=(i, median_val),           
                xytext=(0, median_offset),     # <-- Now controlled by the parameter
                textcoords="offset points",   
                ha='center',
                va='bottom',
                fontsize=median_size,
                color='black',
                # fontweight='bold'
            )

    ax.grid(False)
    fig.tight_layout()
    if file:
        plt.savefig(file, bbox_inches='tight', pad_inches=0.01, dpi=300)
        plt.close(fig)
    else:
        plt.show()

In [48]:
median_values = df1.groupby('label')['word_count'].median().sort_values(ascending=False)
median_order = median_values.index.tolist()

plot_boxplot(
    df1,
    'label',
    'word_count',
    'Merge rate',
    '# of words',
    'label',
    median_order,
    median_values,
    log_scale=False,
    y_ticks=[100, 1000, 2000, 3000, 4000, 5000],
    file='../figs/word_count_plot.pdf',
    median_size=11,
    median_offset=7
)

In [49]:
from scipy.stats import mannwhitneyu

# Example: comparing word_count for Label A and Label B
v1 = df2.loc[df2['label']=='Increase', 'word_count']
v2 = df2.loc[df2['label']=='Decrease', 'word_count']

mannUandCliffdelta(v1, v2), v1.median(), v2.median()

Cliff's delta: small, d=0.22475369458128078
Mann-Whitney-U-test: u=994.5 p=0.09159826385129348


((994.5, 0.09159826385129348, 0.22475369458128078, 'small'), 1182.0, 838.0)

In [50]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_boxplot_final(
    data, x, y, hue, xlab, ylab, median_order, 
    file=None, log_scale=True, y_ticks=None, ylim=None, 
    median_size=7, fig_size=(4.5, 3.0), box_width=0.6,
    median_offset=3  # <-- New parameter: controls vertical shift of text
):
    fig, ax = plt.subplots(figsize=fig_size) 

    sns.boxplot(
        data=data, x=x, y=y, order=median_order, hue=hue,
        showfliers=False, showcaps=False, patch_artist=True,
        palette='Set2', width=box_width, ax=ax,
        medianprops={"linewidth": 1, "color": "grey"}
    )

    plt.xlabel(xlab)
    plt.ylabel(ylab)
    
    # Legend logic
    ax.legend(
        loc='upper right', 
        bbox_to_anchor=(0.99, 0.99), 
        ncol=1, 
        fontsize=median_size,
        frameon=True,
        framealpha=0.9,
        handletextpad=0.5
    )

    if ylim: ax.set_ylim(ylim)
    if log_scale:
        ax.set_yscale("symlog")
        if y_ticks:
            ax.set_yticks(y_ticks)
            ax.set_yticklabels(y_ticks)
    elif y_ticks:
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(y_ticks)

    # Calculation for grouped median positions
    full_medians = data.groupby([x, hue])[y].median()
    hue_labels = data[hue].unique()
    n_hues = len(hue_labels)
    
    # This logic calculates the center of each hue's box within the category
    offsets = np.linspace(-box_width/2 + box_width/(2*n_hues), 
                          box_width/2 - box_width/(2*n_hues), n_hues)

    for i, category in enumerate(median_order):
        for j, label in enumerate(hue_labels):
            try:
                median_val = full_medians.loc[(category, label)]
                x_pos = i + offsets[j]
                
                text_an = f"{median_val:.2f}".replace(".00", "")
                ax.annotate(
                    text=text_an, 
                    xy=(x_pos, median_val),
                    xytext=(0, median_offset), # <-- Applied the parameter here
                    textcoords="offset points",
                    ha='center', 
                    va='bottom',
                    fontsize=median_size,
                    fontweight='bold'
                )
            except KeyError:
                # Handle cases where a specific hue might be missing for a category
                continue 

    ax.grid(False)
    fig.tight_layout()
    
    if file:
        plt.savefig(file, bbox_inches='tight', pad_inches=0.01, dpi=300)
        plt.close()
    else:
        plt.show()

In [51]:
# Assuming df is your original dataframe
h_cols = ["H1", "H2", "H3", "H4", "H5", "H6"]
df_melted = df2.melt(id_vars=['label'], value_vars=h_cols, 
                    var_name='H_Feature', value_name='H_Value')

plot_boxplot_final(
    data=df_melted,
    x='H_Feature',
    y='H_Value',
    hue='label',
    xlab='Header Type',
    ylab='Total count',
    median_order=h_cols,
    log_scale=False,
    y_ticks=[0, 10, 20, 30, 40],
    file='../figs/header_plot.pdf',
    median_size=7,
    ylim=-1,
    fig_size=(3.5, 2.3),
    box_width=0.9,
    median_offset=10
)

In [52]:
import pandas as pd
from scipy.stats import mannwhitneyu

def analyze_heuristics_significance(df):
    # 1. Define the two groups based on merge improvement
    increased = df[df['merge_impro_pct'] > 0]
    decreased = df[df['merge_impro_pct'] < 0]
    
    heuristics = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
    results = []

    for h in heuristics:
        group_inc = increased[h].dropna()
        group_dec = decreased[h].dropna()
        
        # 2. Perform Mann-Whitney U Test
        # alternative='two-sided' checks if they are different in either direction
        _, p, d, size = mannUandCliffdelta(group_inc, group_dec)
        
        # 3. Calculate Effect Size (CLES / Probability of Superiority)
        # P(inc > dec) = U / (n1 * n2)
        # cles = stat / (len(group_inc) * len(group_dec))
        
        results.append({
            'heuristic': h,
            'median_inc': group_inc.median(),
            'median_dec': group_dec.median(),
            # 'U_Stat': stat,
            'p-value': p,
            'delta': d,
            'effect_size': size,
            'significant': p < 0.05
        })

    # Convert to DataFrame for a clean output table
    results_df = pd.DataFrame(results)
    return results_df

# Usage
results = analyze_heuristics_significance(df2)
# print(results.to_string(index=False))

Cliff's delta: small, d=0.20751231527093597
Mann-Whitney-U-test: u=980.5 p=0.11691332611550681
Cliff's delta: negligible, d=0.062192118226600986
Mann-Whitney-U-test: u=862.5 p=0.642424812086186
Cliff's delta: small, d=0.18472906403940886
Mann-Whitney-U-test: u=962.0 p=0.16249380701158522
Cliff's delta: negligible, d=0.008004926108374385
Mann-Whitney-U-test: u=818.5 p=0.9403605805850123
Cliff's delta: negligible, d=0.0
Mann-Whitney-U-test: u=812.0 p=1.0
Cliff's delta: negligible, d=0.0
Mann-Whitney-U-test: u=812.0 p=1.0


In [53]:
results

,heuristic,median_inc,median_dec,p-value,delta,effect_size,significant
0,H1,6.5,3.0,0.116913,0.207512,small,False
1,H2,9.0,8.0,0.642425,0.062192,negligible,False
2,H3,12.0,9.0,0.162494,0.184729,small,False
3,H4,0.0,0.0,0.940361,0.008005,negligible,False
4,H5,0.0,0.0,1.000000,0.000000,negligible,False
5,H6,0.0,0.0,1.000000,0.000000,negligible,False
